# RAG-Powered Academic Research Assistant

**Phase 2 — Build & Evaluate the RAG Pipeline**

---

This notebook builds and evaluates a Retrieval-Augmented Generation (RAG) pipeline over a
local corpus of academic papers on **Artificial Intelligence and Machine Learning in Healthcare**
(medical diagnosis and prediction, medical imaging and computer vision, deep learning,
bioinformatics and multi-omics, image classification / detection / segmentation,
explainable AI, clinical decision support, and healthcare data analysis).

### Reading note on reported numbers

Every quantitative statement in this notebook — document counts, page counts, chunk counts,
retrieval results, and evaluation rates — is **rendered from variables computed by the cells
above it**. Nothing is typed in by hand. Sections that require human judgement (retrieval
relevance and answer grounding labels) are explicitly left blank until a human fills them in,
and the summary statistics are computed only over the labels that have actually been assigned.

## 1. Setup

### What this project does

The assistant answers research questions about a private collection of academic PDFs. Instead of
relying on the language model's parametric memory — which cannot cite a specific paper and is prone
to fabricating references — the system retrieves the passages that actually exist in the corpus and
requires the model to answer from them, citing the source file and page for each claim.

### Architecture

```text
PDF corpus
     |
     v
Page-level extraction (PyMuPDF)          -> metadata: source, page, file_type
     |
     v
Chunking (RecursiveCharacterTextSplitter) -> metadata preserved on every chunk
     |
     v
Embedding (sentence-transformers/all-MiniLM-L6-v2, local, normalized)
     |
     v
Chroma vector store, persisted to models/vectorstore/
     |
     v
Similarity retrieval (top-k)  ->  prompt assembly  ->  LLM  ->  grounded answer + citations
```

The offline half (extraction, chunking, embedding, indexing) runs **only in this notebook**. The
backend loads the persisted Chroma collection and performs retrieval and generation only — it never
re-reads PDFs or recomputes embeddings on a user request.

### Purpose of this notebook

1. Inspect the corpus and record what is actually extractable.
2. Choose and justify a chunking strategy.
3. Build and persist the vector store.
4. Implement retrieval, the prompt template, and the answering function with citation grounding.
5. State the track (Core vs Extended) for the vision component.
6. Evaluate retrieval relevance and answer grounding, and analyse observed failures.
7. Export a backend-ready artefact and verify it reloads from scratch.

In [40]:
# --- Standard library -------------------------------------------------------
import os
import re
import json
import sys
import time
import random
import platform
import statistics
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import warnings
warnings.filterwarnings("ignore", message=".*resume_download.*")
warnings.filterwarnings("ignore", message=".*TypedStorage.*")

from IPython.display import Markdown, display


def show_md(text: str) -> None:
    """Render a string as Markdown output (used for computed summaries)."""
    display(Markdown(text))


# --- Third-party ------------------------------------------------------------
# Each import is checked separately so a missing package produces an actionable
# message instead of an opaque traceback halfway down the notebook.
MISSING: List[str] = []

try:
    import pandas as pd
except ImportError:
    MISSING.append("pandas")

try:
    import PyMuPDF  # PyMuPDF - the installable package is "pymupdf", the import name is "fitz"
except ImportError:
    MISSING.append("pymupdf")

try:
    from langchain_core.documents import Document
except ImportError:
    try:
        from langchain.schema import Document  # older LangChain
    except ImportError:
        MISSING.append("langchain-core")

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    try:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
    except ImportError:
        MISSING.append("langchain-text-splitters")

if MISSING:
    print("Missing required packages:", ", ".join(MISSING))
    print("Install them with:\n")
    print("    pip install " + " ".join(MISSING))
else:
    print("Core imports OK.")

Missing required packages: pymupdf
Install them with:

    pip install pymupdf


In [41]:
# --- Project paths ----------------------------------------------------------
# Resolved relative to the notebook so the same code works on Windows, macOS and Linux.
# No drive letters, no forward/backslash assumptions.

def find_project_root(start: Optional[Path] = None) -> Path:
    """
    Walk upwards from the notebook looking for a directory that looks like the
    project root (contains a 'notebooks' folder, or a git/requirements marker).
    Falls back to the parent of the notebook directory.
    """
    here = (start or Path.cwd()).resolve()
    candidates = [here] + list(here.parents)
    markers = ("notebooks", ".git", "requirements.txt", "pyproject.toml", "src", "app", "backend")
    for path in candidates:
        hits = sum(1 for m in markers if (path / m).exists())
        if hits >= 2 or (path / "notebooks").is_dir():
            return path
    return here.parent if here.name.lower() == "notebooks" else here


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
VECTORSTORE_DIR = MODELS_DIR / "vectorstore"
CONFIG_PATH = VECTORSTORE_DIR / "config.json"
ARTIFACTS_DIR = PROJECT_ROOT / "reports"        # inspection tables, eval CSVs

MODELS_DIR.mkdir(parents=True, exist_ok=True)
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("DATA_DIR       :", DATA_DIR, "(exists:", DATA_DIR.exists(), ")")
print("VECTORSTORE_DIR:", VECTORSTORE_DIR)
print("ARTIFACTS_DIR  :", ARTIFACTS_DIR)

# Show what actually lives under the project root, so the corpus location is
# confirmed from the filesystem rather than assumed.
print("\nTop-level contents of PROJECT_ROOT:")
for entry in sorted(PROJECT_ROOT.iterdir()):
    if entry.name.startswith((".", "__")):
        continue
    print(f"  {'[dir] ' if entry.is_dir() else '[file]'} {entry.name}")

PROJECT_ROOT   : C:\Users\nada4\Downloads\rag-assistant-project
DATA_DIR       : C:\Users\nada4\Downloads\rag-assistant-project\data (exists: True )
VECTORSTORE_DIR: C:\Users\nada4\Downloads\rag-assistant-project\models\vectorstore
ARTIFACTS_DIR  : C:\Users\nada4\Downloads\rag-assistant-project\reports

Top-level contents of PROJECT_ROOT:
  [dir]  data
  [dir]  models
  [dir]  notebooks
  [dir]  reports


In [42]:
# Load environment variables
from dotenv import load_dotenv
import os

PROJECT_ROOT = Path.cwd().parent

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.environ.get("GROQ_API_KEY", "").strip()

print("Project root:", PROJECT_ROOT)
print("GROQ_API_KEY loaded:", bool(api_key))

Project root: c:\Users\nada4\Downloads\rag-assistant-project
GROQ_API_KEY loaded: True


In [43]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

LLM_AVAILABLE = False

try:
    response = llm.invoke("Reply with the single word: ready")
    print(response.content)
    LLM_AVAILABLE = True
    print("LLM is working!")
except Exception as e:
    print(type(e).__name__, e)

print("LLM_AVAILABLE =", LLM_AVAILABLE)

ready
LLM is working!
LLM_AVAILABLE = True


In [44]:
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm

# --------------------------------------------------
# Paths
# --------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PDF_DIR = DATA_DIR / "pdfs"

PDF_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Load metadata
# --------------------------------------------------
metadata = pd.read_csv(DATA_DIR / "metadata.csv")

print(f"Found {len(metadata)} papers in metadata.")
print(f"Downloading PDFs to: {PDF_DIR}")

# --------------------------------------------------
# Download PDFs
# --------------------------------------------------
failed = []

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata),
    desc="Downloading PDFs"
):
    url = row["url"]
    filename = row["filename"]

    output_path = PDF_DIR / filename

    # Skip already downloaded files
    if output_path.exists():
        continue

    try:
        response = requests.get(
            url,
            timeout=60,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        response.raise_for_status()

        output_path.write_bytes(response.content)

    except Exception as e:
        failed.append({
            "filename": filename,
            "url": url,
            "error": str(e)
        })

print("\nDownload complete.")
print(f"Downloaded/available: {len(list(PDF_DIR.glob('*.pdf')))}")
print(f"Failed: {len(failed)}")

if failed:
    print("\nFailed downloads:")
    for item in failed:
        print(item["filename"], "->", item["error"])

Found 24 papers in metadata.



Download complete.
Downloaded/available: 24
Failed: 0


In [45]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print("PDFs found:", len(pdf_files))
for pdf in pdf_files:
    print(pdf.name)

PDFs found: 24
paper_001.pdf
paper_002.pdf
paper_003.pdf
paper_004.pdf
paper_005.pdf
paper_006.pdf
paper_007.pdf
paper_008.pdf
paper_009.pdf
paper_010.pdf
paper_011.pdf
paper_012.pdf
paper_013.pdf
paper_014.pdf
paper_015.pdf
paper_016.pdf
paper_017.pdf
paper_018.pdf
paper_019.pdf
paper_020.pdf
paper_021.pdf
paper_022.pdf
paper_023.pdf
paper_024.pdf


In [46]:
# --- Configuration ----------------------------------------------------------
# These variables are the single source of truth. config.json, the reported
# summaries and the backend contract are all generated from them.

CHUNK_SIZE       = 1000
CHUNK_OVERLAP    = 200
SEPARATORS       = ["\n\n", "\n", ". ", " ", ""]

EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
NORMALIZE_EMBEDDINGS = True          # cosine-style similarity on unit vectors
VECTOR_STORE     = "Chroma"
COLLECTION_NAME  = "academic_rag"
TOP_K            = 5

# Text-extraction quality thresholds used by the corpus inspection in 2.1.
MIN_CHARS_PER_PAGE_FOR_TEXT = 100    # below this a page is treated as image-only
OCR_PAGE_RATIO_THRESHOLD    = 0.50   # >50% low-text pages => document flagged for OCR

# LLM (optional; retrieval works without it). Key is read from the environment only.
LLM_PROVIDER     = "groq"
LLM_MODEL        = "openai/gpt-oss-120b"
LLM_TEMPERATURE  = 0
LLM_MAX_TOKENS   = 1024

# Rebuild control: set to True to force a fresh index even if one is persisted.
FORCE_REBUILD    = False

# Vision component: Core Track unless an image dataset exists AND this is set True.
EXTENDED_TRACK   = False

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
try:
    import numpy as np
    np.random.seed(RANDOM_SEED)
except ImportError:
    np = None
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.use_deterministic_algorithms(False)  # MiniLM inference is deterministic in practice
except ImportError:
    torch = None

print("Configuration set.")

Configuration set.


In [47]:
# --- Reproducibility banner -------------------------------------------------
def _version(module_name: str) -> str:
    try:
        from importlib.metadata import version
        return version(module_name)
    except Exception:
        return "not installed"


ENV_INFO = {
    "python": sys.version.split()[0],
    "platform": f"{platform.system()} {platform.release()}",
    "pandas": _version("pandas"),
    "pymupdf": _version("pymupdf"),
    "langchain-core": _version("langchain-core"),
    "langchain-text-splitters": _version("langchain-text-splitters"),
    "langchain-huggingface": _version("langchain-huggingface"),
    "langchain-chroma": _version("langchain-chroma"),
    "chromadb": _version("chromadb"),
    "sentence-transformers": _version("sentence-transformers"),
    "torch": _version("torch"),
}

print("Environment")
print("-" * 52)
for key, value in ENV_INFO.items():
    print(f"  {key:<26} {value}")

print("\nPipeline configuration")
print("-" * 52)
for key, value in [
    ("embedding_model", EMBEDDING_MODEL),
    ("normalize_embeddings", NORMALIZE_EMBEDDINGS),
    ("chunk_size", CHUNK_SIZE),
    ("chunk_overlap", CHUNK_OVERLAP),
    ("vector_store", VECTOR_STORE),
    ("collection_name", COLLECTION_NAME),
    ("top_k", TOP_K),
    ("random_seed", RANDOM_SEED),
]:
    print(f"  {key:<26} {value}")

Environment
----------------------------------------------------
  python                     3.11.9
  platform                   Windows 10
  pandas                     3.0.5
  pymupdf                    1.28.2
  langchain-core             1.6.3
  langchain-text-splitters   1.1.2
  langchain-huggingface      1.2.2
  langchain-chroma           1.1.0
  chromadb                   1.5.9
  sentence-transformers      6.0.1
  torch                      2.14.0

Pipeline configuration
----------------------------------------------------
  embedding_model            sentence-transformers/all-MiniLM-L6-v2
  normalize_embeddings       True
  chunk_size                 1000
  chunk_overlap              200
  vector_store               Chroma
  collection_name            academic_rag
  top_k                      5
  random_seed                42


## 2.1 Load & Inspect

The corpus directory is **discovered, not assumed**: the code searches the project for directories
containing PDFs and picks the one with the most files, printing every candidate it found so the
choice is auditable. Each PDF is then opened with PyMuPDF and characterised page by page:
page count, extracted character count, how many pages fall below the text threshold, whether the
document is extractable at all, whether it likely needs OCR, and any parsing error.

In [48]:
# --- Locate the PDF corpus --------------------------------------------------
def discover_pdf_directory(root: Path) -> Tuple[Optional[Path], "pd.DataFrame"]:
    """
    Find directories under `root` that contain PDF files and return the richest one.
    Returns (chosen_directory, table_of_all_candidates).
    """
    skip = {".git", ".venv", "venv", "env", "node_modules", "__pycache__", ".ipynb_checkpoints"}
    counts: Dict[Path, int] = {}

    for pdf in root.rglob("*.pdf"):
        if any(part in skip for part in pdf.parts):
            continue
        counts[pdf.parent] = counts.get(pdf.parent, 0) + 1

    if not counts:
        return None, pd.DataFrame(columns=["directory", "pdf_count"])

    rows = [
        {"directory": str(d.relative_to(root)) if d != root else ".", "pdf_count": n}
        for d, n in counts.items()
    ]
    table = pd.DataFrame(rows).sort_values("pdf_count", ascending=False).reset_index(drop=True)
    chosen = max(counts, key=counts.get)
    return chosen, table


PDF_DIR, PDF_DIR_CANDIDATES = discover_pdf_directory(PROJECT_ROOT)

if PDF_DIR is None:
    print("No PDF files found anywhere under", PROJECT_ROOT)
    print("Place the Phase 1 corpus under data/ (any sub-folder) and re-run this cell.")
    PDF_FILES: List[Path] = []
else:
    PDF_FILES = sorted(p for p in PDF_DIR.rglob("*.pdf"))
    print("Candidate directories containing PDFs:")
    display(PDF_DIR_CANDIDATES)
    print("\nSelected corpus directory:", PDF_DIR)
    print("PDF files found            :", len(PDF_FILES))

# Also record any non-PDF files sitting in the same directory, so the
# "file formats" statement later is based on what is actually there.
OTHER_FILES = []
if PDF_DIR is not None:
    OTHER_FILES = [p for p in PDF_DIR.rglob("*") if p.is_file() and p.suffix.lower() != ".pdf"]
    if OTHER_FILES:
        exts = sorted({p.suffix.lower() or "(no extension)" for p in OTHER_FILES})
        print("Non-PDF files also present :", len(OTHER_FILES), "->", ", ".join(exts))

Candidate directories containing PDFs:


,directory,pdf_count
0,data\pdfs,24



Selected corpus directory: c:\Users\nada4\Downloads\rag-assistant-project\data\pdfs
PDF files found            : 24


In [49]:
# --- Per-document inspection ------------------------------------------------
def inspect_documents(pdf_paths: List[Path]) -> "pd.DataFrame":
    """
    Open every PDF and record extraction quality.

    Columns: filename, pages, characters, low_text_pages, text_extractable, needs_ocr, error
    A document is `text_extractable` when it yields a non-trivial amount of text overall;
    it is flagged `needs_ocr` when most of its pages are effectively empty (image-only),
    which is the usual signature of a scanned paper.
    """
    records = []
    for path in pdf_paths:
        record = {
            "filename": path.name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "pages": 0,
            "characters": 0,
            "low_text_pages": 0,
            "text_extractable": False,
            "needs_ocr": False,
            "error": "",
        }
        try:
            with fitz.open(path) as doc:
                if doc.is_encrypted and not doc.authenticate(""):
                    raise RuntimeError("encrypted / password protected")
                record["pages"] = doc.page_count
                total_chars = 0
                low_text = 0
                for page in doc:
                    text = page.get_text("text") or ""
                    n = len(text.strip())
                    total_chars += n
                    if n < MIN_CHARS_PER_PAGE_FOR_TEXT:
                        low_text += 1
                record["characters"] = total_chars
                record["low_text_pages"] = low_text
                if record["pages"]:
                    ocr_ratio = low_text / record["pages"]
                    record["needs_ocr"] = ocr_ratio > OCR_PAGE_RATIO_THRESHOLD
                    record["text_extractable"] = (
                        total_chars >= MIN_CHARS_PER_PAGE_FOR_TEXT and not record["needs_ocr"]
                    )
        except Exception as exc:                      # corrupt, encrypted, truncated...
            record["error"] = f"{type(exc).__name__}: {exc}"
        records.append(record)

    return pd.DataFrame.from_records(records)


inspection_df = inspect_documents(PDF_FILES)
if not inspection_df.empty:
    display(
        inspection_df[
            ["filename", "pages", "characters", "low_text_pages",
             "text_extractable", "needs_ocr", "error"]
        ]
    )
    inspection_df.to_csv(ARTIFACTS_DIR / "corpus_inspection.csv", index=False)
    print("Saved:", ARTIFACTS_DIR / "corpus_inspection.csv")
else:
    print("Inspection table is empty - no PDFs were located.")

,filename,pages,characters,low_text_pages,text_extractable,needs_ocr,error
0,paper_001.pdf,38,217366,0,True,False,
1,paper_002.pdf,27,149251,0,True,False,
2,paper_003.pdf,52,86050,0,True,False,
3,paper_004.pdf,34,57494,1,True,False,
4,paper_005.pdf,1,4,1,False,True,
5,paper_006.pdf,1,4,1,False,True,
6,paper_007.pdf,1,4,1,False,True,
7,paper_008.pdf,1,4,1,False,True,
8,paper_009.pdf,1,4,1,False,True,
9,paper_010.pdf,1,4,1,False,True,


Saved: C:\Users\nada4\Downloads\rag-assistant-project\reports\corpus_inspection.csv


In [50]:
# --- Corpus statistics (computed, never hard-coded) -------------------------
def corpus_stats(df: "pd.DataFrame") -> Dict[str, Any]:
    if df.empty:
        return {
            "n_documents": 0, "total_pages": 0, "total_characters": 0,
            "n_parsed": 0, "n_needs_ocr": 0, "n_failed": 0,
            "mean_pages": 0.0, "mean_chars_per_page": 0.0, "formats": {},
        }
    failed = df["error"].astype(bool)
    stats = {
        "n_documents": int(len(df)),
        "total_pages": int(df["pages"].sum()),
        "total_characters": int(df["characters"].sum()),
        "n_parsed": int((~failed).sum()),
        "n_needs_ocr": int(df["needs_ocr"].sum()),
        "n_failed": int(failed.sum()),
        "mean_pages": float(df["pages"].mean()),
        "mean_chars_per_page": float(
            df["characters"].sum() / df["pages"].sum()
        ) if df["pages"].sum() else 0.0,
    }
    formats = {".pdf": int(len(df))}
    for path in OTHER_FILES:
        key = path.suffix.lower() or "(no extension)"
        formats[key] = formats.get(key, 0) + 1
    stats["formats"] = formats
    return stats


CORPUS_STATS = corpus_stats(inspection_df)
for key, value in CORPUS_STATS.items():
    print(f"{key:<22} {value}")

n_documents            24
total_pages            489
total_characters       1348159
n_parsed               24
n_needs_ocr            12
n_failed               0
mean_pages             20.375
mean_chars_per_page    2756.971370143149
formats                {'.pdf': 24}


In [51]:
# --- Dataset summary, rendered from the numbers computed above --------------
def render_dataset_summary(stats: Dict[str, Any], df: "pd.DataFrame") -> str:
    if stats["n_documents"] == 0:
        return "### Dataset Summary\n\nNo documents were found, so no summary can be produced."

    fmt_line = ", ".join(f"`{ext}` x {n}" for ext, n in sorted(stats["formats"].items()))
    ocr_docs = df.loc[df["needs_ocr"], "filename"].tolist()
    failed_docs = df.loc[df["error"].astype(bool), ["filename", "error"]].values.tolist()

    lines = [
        "### Dataset Summary",
        "",
        f"The corpus contains **{stats['n_documents']} PDF documents** with "
        f"**{stats['total_pages']} total pages** and {stats['total_characters']:,} extracted "
        f"characters (mean {stats['mean_pages']:.1f} pages per document, "
        f"{stats['mean_chars_per_page']:.0f} characters per page).",
        "",
        f"File formats present in the corpus directory: {fmt_line}.",
        "",
        f"**{stats['n_parsed']}** documents were parsed without error, "
        f"**{stats['n_failed']}** failed to parse, and "
        f"**{stats['n_needs_ocr']}** contained insufficient extractable text and would require OCR "
        f"(more than {int(OCR_PAGE_RATIO_THRESHOLD * 100)}% of their pages fell below "
        f"{MIN_CHARS_PER_PAGE_FOR_TEXT} characters).",
    ]
    if ocr_docs:
        lines += ["", "Documents flagged for OCR:", ""]
        lines += [f"- `{name}`" for name in ocr_docs]
    if failed_docs:
        lines += ["", "Documents that failed to parse:", ""]
        lines += [f"- `{name}` — {err}" for name, err in failed_docs]
    if not ocr_docs and not failed_docs:
        lines += ["", "No OCR-only or unparseable documents were detected in this corpus."]

    low = df.loc[(~df["needs_ocr"]) & (df["low_text_pages"] > 0)]
    if len(low):
        lines += [
            "",
            f"A further **{len(low)}** otherwise-readable documents contain "
            f"{int(low['low_text_pages'].sum())} individual low-text pages — typically full-page "
            "figures, scanned plates, or blank separator pages. These pages are dropped before "
            "chunking rather than embedded as near-empty vectors.",
        ]
    return "\n".join(lines)


show_md(render_dataset_summary(CORPUS_STATS, inspection_df))

### Dataset Summary

The corpus contains **24 PDF documents** with **489 total pages** and 1,348,159 extracted characters (mean 20.4 pages per document, 2757 characters per page).

File formats present in the corpus directory: `.pdf` x 24.

**24** documents were parsed without error, **0** failed to parse, and **12** contained insufficient extractable text and would require OCR (more than 50% of their pages fell below 100 characters).

Documents flagged for OCR:

- `paper_005.pdf`
- `paper_006.pdf`
- `paper_007.pdf`
- `paper_008.pdf`
- `paper_009.pdf`
- `paper_010.pdf`
- `paper_011.pdf`
- `paper_012.pdf`
- `paper_015.pdf`
- `paper_016.pdf`
- `paper_017.pdf`
- `paper_022.pdf`

A further **1** otherwise-readable documents contain 1 individual low-text pages — typically full-page figures, scanned plates, or blank separator pages. These pages are dropped before chunking rather than embedded as near-empty vectors.

The summary rendered above is generated by `render_dataset_summary()` directly from
`CORPUS_STATS` and the inspection DataFrame. If the corpus changes, re-running the notebook
updates it; there are no transcribed numbers to fall out of date.

Formatting characteristics worth noting for the chunking decision that follows: academic PDFs are
typeset in two columns, which makes PyMuPDF's reading order interleave column fragments on some
pages; running headers, page numbers, and DOI footers repeat on every page; references sections
produce long runs of citation strings with little semantic content; and tables lose their row/column
structure, arriving as whitespace-separated number sequences. The cleaning step below removes the
most damaging of these artefacts before chunking.

In [52]:
# --- Page-level Document objects with citation metadata ---------------------
HEADER_FOOTER_PATTERNS = [
    re.compile(r"^\s*\d{1,4}\s*$"),                              # bare page number line
    re.compile(r"^\s*Page\s+\d+\s*(of\s+\d+)?\s*$", re.I),
    re.compile(r"^\s*(https?://|doi:|DOI\s)", re.I),
    re.compile(r"^\s*Downloaded from\b", re.I),
]


def clean_page_text(text: str) -> str:
    """Light, conservative cleaning: de-hyphenate line breaks, drop repeated
    header/footer lines, and collapse runaway whitespace. Nothing that would
    alter the meaning of a sentence."""
    text = text.replace("\x00", " ")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)      # word split across lines
    kept = []
    for line in text.split("\n"):
        if any(p.match(line) for p in HEADER_FOOTER_PATTERNS):
            continue
        kept.append(line)
    text = "\n".join(kept)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_documents(pdf_paths: List[Path], df: "pd.DataFrame") -> List["Document"]:
    """
    Produce one LangChain Document per usable page.

    Metadata attached to every page (and therefore to every chunk derived from it):
        source    -> filename, used for citations
        page      -> 1-based page number, used for citations
        file_type -> "pdf"
        chars     -> cleaned character count, useful for later diagnostics
    """
    failed = set(df.loc[df["error"].astype(bool), "filename"]) if not df.empty else set()
    docs: List[Document] = []
    skipped_pages = 0

    for path in pdf_paths:
        if path.name in failed:
            continue
        try:
            with fitz.open(path) as pdf:
                for index, page in enumerate(pdf):
                    raw = page.get_text("text") or ""
                    text = clean_page_text(raw)
                    if len(text) < MIN_CHARS_PER_PAGE_FOR_TEXT:
                        skipped_pages += 1          # image-only / blank page
                        continue
                    docs.append(
                        Document(
                            page_content=text,
                            metadata={
                                "source": path.name,
                                "page": index + 1,
                                "file_type": "pdf",
                                "chars": len(text),
                            },
                        )
                    )
        except Exception as exc:
            print(f"  ! skipped {path.name}: {type(exc).__name__}: {exc}")

    print(f"Page documents created : {len(docs)}")
    print(f"Pages skipped (low text): {skipped_pages}")
    return docs


page_documents = load_documents(PDF_FILES, inspection_df)

if page_documents:
    sample = page_documents[0]
    print("\nExample page document")
    print("  metadata:", sample.metadata)
    print("  preview :", sample.page_content[:300].replace("\n", " "), "...")

Page documents created : 476
Pages skipped (low text): 13

Example page document
  metadata: {'source': 'paper_001.pdf', 'page': 1, 'file_type': 'pdf', 'chars': 3869}
  preview : A Survey on Deep Learning in Medical Image Analysis Geert Litjens, Thijs Kooi, Babak Ehteshami Bejnordi, Arnaud Arindra Adiyoso Setio, Francesco Ciompi, Mohsen Ghafoorian, Jeroen A.W.M. van der Laak, Bram van Ginneken, Clara I. S´anchez Diagnostic Image Analysis Group Radboud University Medical Cent ...


## 2.2 Chunking Strategy

Page-level documents are too coarse to retrieve: a single page of a paper mixes several ideas, and
embedding it produces an averaged vector that matches many queries weakly and none strongly. The
pages are therefore split with `RecursiveCharacterTextSplitter`, which tries the separators in order
and only falls back to a cruder split when the finer one cannot satisfy the size limit — so it breaks
at paragraph boundaries where possible, then at line breaks, then sentences, and only splits mid-word
as a last resort.

In [53]:
def chunk_documents(
    docs: List["Document"],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: Optional[List[str]] = None,
) -> List["Document"]:
    """
    Split page documents into overlapping chunks.

    `RecursiveCharacterTextSplitter` copies the parent document's metadata onto every
    child chunk, so `source` and `page` survive the split. A `chunk_id` is added
    afterwards to give each chunk a stable identity for evaluation and debugging.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators or SEPARATORS,
        length_function=len,
        add_start_index=True,      # records where in the page the chunk began
    )
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata.setdefault("source", "unknown")
        chunk.metadata.setdefault("page", -1)
    return chunks


chunks = chunk_documents(page_documents)
print(f"Chunks generated: {len(chunks)}")

Chunks generated: 1801


In [54]:
# --- Chunk statistics -------------------------------------------------------
def chunk_statistics(page_docs: List["Document"], chunk_docs: List["Document"]) -> Dict[str, Any]:
    lengths = [len(c.page_content) for c in chunk_docs]
    if not lengths:
        return {}
    return {
        "n_page_documents": len(page_docs),
        "n_chunks": len(chunk_docs),
        "chunks_per_page": len(chunk_docs) / len(page_docs) if page_docs else 0.0,
        "min_length": min(lengths),
        "max_length": max(lengths),
        "mean_length": statistics.mean(lengths),
        "median_length": statistics.median(lengths),
        "stdev_length": statistics.pstdev(lengths),
        "n_unique_sources": len({c.metadata.get("source") for c in chunk_docs}),
    }


CHUNK_STATS = chunk_statistics(page_documents, chunks)

print("Chunking results")
print("-" * 46)
print(f"  Original page documents : {CHUNK_STATS.get('n_page_documents', 0)}")
print(f"  Generated chunks        : {CHUNK_STATS.get('n_chunks', 0)}")
print(f"  Chunks per page (mean)  : {CHUNK_STATS.get('chunks_per_page', 0):.2f}")
print(f"  Distinct source files   : {CHUNK_STATS.get('n_unique_sources', 0)}")
print("-" * 46)
print(f"  Minimum chunk length    : {CHUNK_STATS.get('min_length', 0)}")
print(f"  Maximum chunk length    : {CHUNK_STATS.get('max_length', 0)}")
print(f"  Mean chunk length       : {CHUNK_STATS.get('mean_length', 0):.1f}")
print(f"  Median chunk length     : {CHUNK_STATS.get('median_length', 0):.1f}")
print(f"  Std. dev. of length     : {CHUNK_STATS.get('stdev_length', 0):.1f}")

Chunking results
----------------------------------------------
  Original page documents : 476
  Generated chunks        : 1801
  Chunks per page (mean)  : 3.78
  Distinct source files   : 12
----------------------------------------------
  Minimum chunk length    : 16
  Maximum chunk length    : 1000
  Mean chunk length       : 856.7
  Median chunk length     : 950.0
  Std. dev. of length     : 201.5


In [55]:
# --- Inspect example chunks -------------------------------------------------
def preview_chunks(chunk_docs: List["Document"], n: int = 3, seed: int = RANDOM_SEED) -> None:
    if not chunk_docs:
        print("No chunks to preview.")
        return
    rng = random.Random(seed)
    picks = rng.sample(range(len(chunk_docs)), k=min(n, len(chunk_docs)))
    for idx in picks:
        c = chunk_docs[idx]
        print("=" * 78)
        print(f"chunk_id={c.metadata['chunk_id']}  source={c.metadata['source']}  "
              f"page={c.metadata['page']}  length={len(c.page_content)}")
        print("-" * 78)
        print(c.page_content[:600].strip())
        if len(c.page_content) > 600:
            print("...")
    print("=" * 78)


preview_chunks(chunks, n=3)

chunk_id=1309  source=paper_020.pdf  page=2  length=590
------------------------------------------------------------------------------
importance. We introduce a percentile variable to limit the number of highlighted words to only the top percent of
features based on their importance scores.
• Attention-based explanations use weights from attention-based models7 to explain model decisions, making it a
model-specific method. We apply the approach outlined by Falaki et al.23 to extract attention weights from our
UmlsBERT model’s [CLS] tokens and recombine subword tokens into words for visualization. We highlight the top
30% of words by attention weights, with the intensity of the highlight defined by the weight value.
chunk_id=228  source=paper_001.pdf  page=33  length=965
------------------------------------------------------------------------------
Int Symp Biomedical Imaging. pp. 791–794.
LeCun, Y., Bottou, L., Bengio, Y., Haﬀner, P., 1998. Gradient-based
learning applied to document 

In [56]:
# --- Verify metadata survived the split -------------------------------------
required_keys = {"source", "page", "file_type", "chunk_id"}
missing = [
    c.metadata.get("chunk_id")
    for c in chunks
    if not required_keys.issubset(c.metadata) or c.metadata.get("page", -1) < 0
]

if chunks and not missing:
    print(f"Metadata check PASSED - all {len(chunks)} chunks carry {sorted(required_keys)}.")
else:
    print(f"Metadata check FAILED for {len(missing)} chunks:", missing[:10])

# Distribution of chunks across source documents (top 10 by chunk count)
if chunks:
    per_source = pd.Series([c.metadata["source"] for c in chunks]).value_counts()
    print("\nChunks per source document (top 10):")
    display(per_source.head(10).to_frame("chunks"))

Metadata check PASSED - all 1801 chunks carry ['chunk_id', 'file_type', 'page', 'source'].

Chunks per source document (top 10):


,chunks
paper_001.pdf,277
paper_018.pdf,224
paper_014.pdf,192
paper_002.pdf,191
paper_024.pdf,178
paper_019.pdf,142
paper_021.pdf,135
paper_023.pdf,135
paper_003.pdf,121
paper_004.pdf,79


### Justification of the chunking configuration

**Chunk size (1000 characters, roughly 150–200 words).** A chunk has to be small enough that its
embedding represents one idea, and large enough that the idea is complete. In the papers in this
corpus, a single argumentative unit — a method description, a result with its metric, a stated
limitation — typically occupies one to two paragraphs, which lands in this range. Much smaller chunks
(200–300 characters) split a claim from the number that supports it, so a query about reported
accuracy retrieves the sentence introducing the experiment but not the figure; much larger chunks
(3000+) dilute the embedding, because a vector averaged over an entire section sits near the centroid
of the corpus and ranks similarly for almost every query.

**Overlap (200 characters, 20%).** Splitting is blind to meaning: a boundary can fall between "the
model achieved" and the sentence that reports what it achieved. Overlap means each boundary region
appears in both neighbouring chunks, so whichever one is retrieved still carries the full statement.
Twenty percent is enough to span a long sentence or a short paragraph without inflating the index
excessively — the redundancy cost shows up directly in the chunks-per-page figure printed above.

**Why this suits academic papers.** The separator order (`"\n\n"`, `"\n"`, `". "`, `" "`, `""`)
matches the structure of the text: paragraph breaks first, then line breaks, then sentence ends.
Papers are written in well-formed paragraphs with explicit section structure, so the splitter usually
succeeds at the first or second separator and rarely has to cut mid-sentence. Keeping one page as one
parent document also means a chunk never spans two pages, which keeps the page citation
unambiguous — every claim can be traced to a specific page of a specific file.

**Effect on retrieval.** Chunk size trades precision against context. Smaller chunks raise precision
(the retrieved text is mostly on-topic) but risk truncating the evidence the model needs; larger
chunks raise recall within a passage but return more irrelevant text per slot, and with a fixed
top-k the prompt fills up with padding. With k = 5 and ~1000-character chunks, the model receives
roughly 5000 characters of context — enough for a synthesis across several papers, still far within
the model's context window.

**This configuration is a reasonable starting point, not an optimum.** It has not been benchmarked
against alternatives on this corpus. The honest claim is that it is a defensible default for
paragraph-structured academic prose; establishing optimality would require a sweep over chunk sizes
measured against a labelled retrieval set, which is out of scope for this phase. Section 2.6 records
whether any observed failure points toward adjusting it.

## 2.3 Embeddings & Vector Store

`sentence-transformers/all-MiniLM-L6-v2` runs locally on CPU: 384-dimensional vectors, no API key,
no per-query cost, and fast enough to embed the whole corpus in this notebook. It is trained for
sentence-level semantic similarity, which is the right objective for matching a question against a
passage. Embeddings are normalised to unit length so that the inner product is cosine similarity,
making scores comparable across chunks of different lengths.

Chroma is used as the store: it persists to a plain directory, keeps arbitrary metadata alongside
each vector, and the same directory is reloaded by the backend with no rebuild.

In [57]:
import warnings
warnings.filterwarnings("ignore", message=".*resume_download.*")
warnings.filterwarnings("ignore", message=".*TypedStorage.*")

# --- Embedding model --------------------------------------------------------
# langchain-huggingface is the current package; fall back to the community
# location for older environments.
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    # If langchain_huggingface is not found, try to install necessary packages.
    # Both langchain-huggingface and langchain-community might be needed.
    print("`langchain-huggingface` or `langchain_community` not found. Attempting to install...")
    %pip install --quiet langchain-huggingface langchain-community
    # After installation, retry the original import sequence
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        from langchain_community.embeddings import HuggingFaceEmbeddings
    print("Required embedding packages installed and imported successfully.")


def build_embeddings(model_name: str = EMBEDDING_MODEL, normalize: bool = NORMALIZE_EMBEDDINGS):
    """Load the local sentence-transformer used for both indexing and querying.
    The same object must be used at query time or the vectors are not comparable."""
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": normalize, "batch_size": 32},
    )


t0 = time.time()
embeddings = build_embeddings()
print(f"Embedding model loaded in {time.time() - t0:.1f}s: {EMBEDDING_MODEL}")

# Confirm the dimensionality from the model itself rather than assuming 384.
probe = embeddings.embed_query("convolutional neural network for medical image segmentation")
EMBEDDING_DIM = len(probe)
print("Embedding dimension:", EMBEDDING_DIM)
if np is not None:
    print("Vector L2 norm     :", round(float(np.linalg.norm(probe)), 4),
          "(≈1.0 confirms normalisation)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded in 3.9s: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Vector L2 norm     : 1.0 (≈1.0 confirms normalisation)


In [58]:
# --- Build / load the Chroma vector store -----------------------------------
try:
    import chromadb
    from langchain_chroma import Chroma
except ImportError:
    print("`chromadb` or `langchain-chroma` not found, or there's a dependency conflict. Attempting to resolve...")

    # First, try to satisfy the google-adk dependency for opentelemetry
    # This might require uninstalling newer versions installed by previous attempts
    %pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc
    %pip install --quiet opentelemetry-api==1.42.1 opentelemetry-sdk==1.42.1 opentelemetry-exporter-otlp-proto-grpc==1.42.1

    # Then install chromadb and langchain-chroma
    %pip install --quiet chromadb langchain-chroma

    # Re-attempt imports
    import chromadb
    from langchain_chroma import Chroma
    print("Required vector store packages installed and imported successfully.")


def _sanitize(meta: Dict[str, Any]) -> Dict[str, Any]:
    """Chroma only accepts str/int/float/bool metadata values."""
    return {k: v for k, v in meta.items() if isinstance(v, (str, int, float, bool))}


def create_vectorstore(
    chunk_docs: List["Document"],
    embedding_function,
    persist_directory: Path = VECTORSTORE_DIR,
    collection_name: str = COLLECTION_NAME,
    batch_size: int = 256,
):
    """
    Embed the chunks and persist them to disk.

    Documents are added in batches to stay under Chroma's per-call limit and to
    give visible progress on larger corpora.
    """
    persist_directory.mkdir(parents=True, exist_ok=True)
    for c in chunk_docs:
        c.metadata = _sanitize(c.metadata)

    store = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_function,
        persist_directory=str(persist_directory),
    )

    total = len(chunk_docs)
    for start in range(0, total, batch_size):
        batch = chunk_docs[start:start + batch_size]
        store.add_documents(batch)
        print(f"  embedded {min(start + batch_size, total)}/{total} chunks", end="\r")
    print()

    # Older LangChain/Chroma versions require an explicit persist call; newer
    # versions persist automatically and no longer expose the method.
    if hasattr(store, "persist"):
        try:
            store.persist()
        except Exception:
            pass
    return store


def open_vectorstore(embedding_function, persist_directory: Path = VECTORSTORE_DIR,
                     collection_name: str = COLLECTION_NAME):
    """Open an existing persisted collection without writing to it."""
    return Chroma(
        collection_name=collection_name,
        embedding_function=embedding_function,
        persist_directory=str(persist_directory),
    )


def collection_count(store) -> int:
    try:
        return store._collection.count()
    except Exception:
        try:
            return len(store.get()["ids"])
        except Exception:
            return -1


existing = open_vectorstore(embeddings)
existing_count = collection_count(existing)

if existing_count > 0 and not FORCE_REBUILD:
    vectorstore = existing
    print(f"Loaded existing collection '{COLLECTION_NAME}' with {existing_count} vectors.")
    print("Set FORCE_REBUILD = True in the configuration cell to rebuild from scratch.")
else:
    if existing_count > 0:
        print(f"FORCE_REBUILD is set - deleting {existing_count} existing vectors.")
        try:
            existing.delete_collection()
        except Exception as exc:
            print("  (could not delete collection:", exc, ")")
    t0 = time.time()
    vectorstore = create_vectorstore(chunks, embeddings)
    print(f"Vector store built in {time.time() - t0:.1f}s")

VECTOR_COUNT = collection_count(vectorstore)
print("Vectors in collection:", VECTOR_COUNT)
print("Persisted to         :", VECTORSTORE_DIR)

Loaded existing collection 'academic_rag' with 1801 vectors.
Set FORCE_REBUILD = True in the configuration cell to rebuild from scratch.
Vectors in collection: 1801
Persisted to         : C:\Users\nada4\Downloads\rag-assistant-project\models\vectorstore


In [59]:
# --- Persist the pipeline configuration -------------------------------------
def save_config(path: Path = CONFIG_PATH) -> Dict[str, Any]:
    """
    Write the contract the backend reads. Every value comes from the live
    configuration variables, so the file cannot drift from what was actually built.
    """
    config = {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "separators": SEPARATORS,
        "embedding_model": EMBEDDING_MODEL,
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "embedding_dimension": EMBEDDING_DIM,
        "vector_store": VECTOR_STORE,
        "collection_name": COLLECTION_NAME,
        "persist_directory": str(VECTORSTORE_DIR.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "top_k": TOP_K,
        "search_type": "similarity",
        "n_source_documents": CORPUS_STATS.get("n_documents", 0),
        "n_pages_indexed": CHUNK_STATS.get("n_page_documents", 0),
        "n_chunks_indexed": CHUNK_STATS.get("n_chunks", 0),
        "n_vectors": VECTOR_COUNT,
        "llm_provider": LLM_PROVIDER,
        "llm_model": LLM_MODEL,
        "llm_temperature": LLM_TEMPERATURE,
        "built_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "python_version": ENV_INFO["python"],
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    return config


CONFIG = save_config()
print("Wrote", CONFIG_PATH, "\n")
print(json.dumps(CONFIG, indent=2))

Wrote C:\Users\nada4\Downloads\rag-assistant-project\models\vectorstore\config.json 

{
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "separators": [
    "\n\n",
    "\n",
    ". ",
    " ",
    ""
  ],
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "normalize_embeddings": true,
  "embedding_dimension": 384,
  "vector_store": "Chroma",
  "collection_name": "academic_rag",
  "persist_directory": "models/vectorstore",
  "top_k": 5,
  "search_type": "similarity",
  "n_source_documents": 24,
  "n_pages_indexed": 476,
  "n_chunks_indexed": 1801,
  "n_vectors": 1801,
  "llm_provider": "groq",
  "llm_model": "openai/gpt-oss-120b",
  "llm_temperature": 0,
  "built_at": "2026-09-18 02:53:39",
  "python_version": "3.11.9"
}


In [60]:
# --- Smoke test: similarity search ------------------------------------------
probe_query = "deep learning for medical image segmentation"
results = vectorstore.similarity_search_with_score(probe_query, k=3)

print(f"Query: {probe_query!r}\n")
for rank, (doc, score) in enumerate(results, start=1):
    print(f"[{rank}] source={doc.metadata.get('source')}  "
          f"page={doc.metadata.get('page')}  distance={score:.4f}")
    print("    ", doc.page_content[:300].replace("\n", " "), "...")
    print()

Query: 'deep learning for medical image segmentation'

[1] source=paper_004.pdf  page=19  distance=0.5321
     Ref. Segmentation Regression Mean absolute error (in months) [94] U-Net VGG16 8.08 [95] U-Net InceptionResNetV2 12.7744 [96] DeepLabV3 MobileNetV1 8.200 [97] U-Net VGG16 9.997 [98] CNN MobileNetV3 6.2 [99] Mask R-CNN VGG19 6.38 Open in a new tab 5. Conclusion We have presented a detailed overview of ...

[2] source=paper_004.pdf  page=4  distance=0.5478
     motivations for this are the availability of computational resources and the resurgence of deep convolutional neural networks. Deep learning techniques are good at observing hidden patterns in images and supporting clinicians in achieving diagnostic perfection. It has proven to be the most effective ...

[3] source=paper_024.pdf  page=20  distance=0.5588
     [16] Syed Muhammad Anwar, Muhammad Majid, Adnan Qayyum, Muhammad Awais, Majdi Alnowami, and Muhammad Khurram Khan. Medical image analysis using convolutional neural n

## 2.4 Retrieval & Prompting

Retrieval returns the top-k chunks by vector similarity, each still carrying its `source` and `page`.
Those two fields are what make citation possible: the filename and page in an answer are read off the
metadata of chunks that were actually retrieved, never produced by the language model on its own.

In [61]:
# --- Retrieval --------------------------------------------------------------
def retrieve_documents(query: str, k: int = TOP_K, store=None) -> List[Tuple["Document", float]]:
    """
    Return the top-k chunks for a query as (document, distance) pairs.
    Lower distance = closer match. Distances are kept so that retrieval
    confidence can be inspected during evaluation.
    """
    store = store or vectorstore
    return store.similarity_search_with_score(query, k=k)


def show_retrieval(query: str, k: int = TOP_K, excerpt: int = 260) -> None:
    hits = retrieve_documents(query, k=k)
    print("=" * 96)
    print("Q:", query)
    print("-" * 96)
    if not hits:
        print("  (no results)")
        return
    for rank, (doc, score) in enumerate(hits, start=1):
        print(f"  [{rank}] {doc.metadata.get('source')}  |  page {doc.metadata.get('page')}  "
              f"|  distance {score:.4f}")
        text = " ".join(doc.page_content.split())
        print(f"      {text[:excerpt]}...")
    print()

In [62]:
# --- Research questions -----------------------------------------------------
# Written to span the subject areas of the corpus. Whether each is actually
# answerable from the collected papers is determined by the retrieval output
# below and by the manual evaluation in 2.6 - it is not assumed here.
RESEARCH_QUESTIONS = [
    "How is artificial intelligence being applied to clinical decision support in healthcare?",
    "What deep learning architectures are used for medical image segmentation?",
    "How are convolutional neural networks applied to medical image classification?",
    "What machine learning methods are used for disease diagnosis and risk prediction?",
    "What explainable AI techniques are used to interpret clinical prediction models?",
    "How is machine learning applied to bioinformatics and genomic data analysis?",
    "What approaches are used to integrate multi-omics data using machine learning?",
    "How are transfer learning and pretrained models used when medical training data is limited?",
    "What evaluation metrics are reported for medical image detection and diagnosis models?",
    "What are the main limitations and challenges of deploying AI models in clinical practice?",
    "How is patient data privacy and bias addressed in healthcare machine learning research?",
    "What role do transformer architectures play in medical imaging or clinical text analysis?",
]

print(f"{len(RESEARCH_QUESTIONS)} research questions defined.")
for i, q in enumerate(RESEARCH_QUESTIONS, start=1):
    print(f"  {i:>2}. {q}")

12 research questions defined.
   1. How is artificial intelligence being applied to clinical decision support in healthcare?
   2. What deep learning architectures are used for medical image segmentation?
   3. How are convolutional neural networks applied to medical image classification?
   4. What machine learning methods are used for disease diagnosis and risk prediction?
   5. What explainable AI techniques are used to interpret clinical prediction models?
   6. How is machine learning applied to bioinformatics and genomic data analysis?
   7. What approaches are used to integrate multi-omics data using machine learning?
   8. How are transfer learning and pretrained models used when medical training data is limited?
   9. What evaluation metrics are reported for medical image detection and diagnosis models?
  10. What are the main limitations and challenges of deploying AI models in clinical practice?
  11. How is patient data privacy and bias addressed in healthcare machine lear

In [63]:
# --- Retrieval over all research questions ----------------------------------
for question in RESEARCH_QUESTIONS:
    show_retrieval(question, k=TOP_K)

Q: How is artificial intelligence being applied to clinical decision support in healthcare?
------------------------------------------------------------------------------------------------
  [1] paper_019.pdf  |  page 5  |  distance 0.5861
      1. Introduction AI has emerged as a transformative force in modern healthcare, particularly through its integration into clinical decision support systems (CDSSs) [1,2]. CDSSs are computational tools designed to assist clinicians in making data-driven decision...
  [2] paper_021.pdf  |  page 6  |  distance 0.5989
      Vince Istvan Madai 16QUEST Center for Responsible Research, Berlin Institute of Health (BIH), Charité Universitätsmedizin Berlin, Germany 17CLAIM—Charité Lab for Artificial Intelligence in Medicine, Charité Universitätsmedizin Berlin, Germany 18School of Compu...
  [3] paper_019.pdf  |  page 4  |  distance 0.6180
      relevant. Keywords: explainable artificial intelligence (XAI), healthcare AI, human-centered AI, healthcare, med

Q: What deep learning architectures are used for medical image segmentation?
------------------------------------------------------------------------------------------------
  [1] paper_004.pdf  |  page 6  |  distance 0.5692
      performance are described. Following that, the finding of models aimed at detecting COVID-19 and predicting child bone age are reviewed in Section 4. And finally, the conclusion is set out. 2. Related Works This section discusses the survey papers on medical i...
  [2] paper_001.pdf  |  page 11  |  distance 0.5725
      pixel/voxel-wise processing of images. We expect that more emphasis will be given to those areas in the near future, for example in the application of multi-stream networks in a fully convolutional fashion. 3.3. Segmentation 3.3.1. Organ and substructure segme...
  [3] paper_001.pdf  |  page 11  |  distance 0.5727
      as such has also seen the widest variety in methodology, including the development of unique CNN-based segmentation architect

In [64]:
# --- Retrieval distance summary --------------------------------------------
# A compact view of how confident retrieval was for each question. Large
# distances are the first signal that a question may fall outside the corpus.
rows = []
for question in RESEARCH_QUESTIONS:
    hits = retrieve_documents(question, k=TOP_K)
    scores = [s for _, s in hits]
    rows.append({
        "question": question[:70] + ("..." if len(question) > 70 else ""),
        "best_distance": round(min(scores), 4) if scores else None,
        "mean_distance": round(statistics.mean(scores), 4) if scores else None,
        "distinct_sources": len({d.metadata.get("source") for d, _ in hits}),
    })

retrieval_summary_df = pd.DataFrame(rows)
display(retrieval_summary_df)

,question,best_distance,mean_distance,distinct_sources
0,How is artificial intelligence being applied t...,0.5861,0.6165,2
1,What deep learning architectures are used for ...,0.5692,0.5761,3
2,How are convolutional neural networks applied ...,0.5497,0.6109,4
3,What machine learning methods are used for dis...,0.9067,0.9398,3
4,What explainable AI techniques are used to int...,0.4833,0.5634,3
5,How is machine learning applied to bioinformat...,0.8715,0.9102,1
6,What approaches are used to integrate multi-om...,0.8001,0.9459,1
7,How are transfer learning and pretrained model...,0.6375,0.7125,3
8,What evaluation metrics are reported for medic...,0.6232,0.7139,3
9,What are the main limitations and challenges o...,0.7199,0.7356,2


### Prompt template

The prompt does three jobs. It fixes the model's role and its evidence rule (answer from the supplied
context only). It presents the retrieved passages already labelled with their filename and page, so
the citation the model writes is a copy of a label in front of it rather than a recollection. And it
supplies an explicit refusal sentence, which gives the model a licence to decline — without one, a
model asked a question its context cannot answer will usually produce a plausible answer anyway.

In [65]:
# --- Prompt template --------------------------------------------------------
PROMPT_TEMPLATE = """You are an academic research assistant working with a corpus of peer-reviewed
papers on artificial intelligence and machine learning in healthcare.

Rules you must follow:
1. Answer using ONLY the research context provided below. Do not use outside knowledge.
2. Do not invent findings, numbers, method names, author names or citations.
3. If the context does not contain enough information to answer, reply with exactly:
   "I could not find sufficient evidence in the provided research corpus."
4. Cite every substantive claim inline using the format [Document: <filename>, Page: <n>],
   copying the filename and page exactly as they appear in the context labels.
5. Where the retrieved papers disagree or address different settings, say so rather than
   merging them into a single claim.
6. Be concise and academically precise. Prefer three to six sentences unless the question
   requires more.

Research Context:
{context}

Question:
{question}

Answer:"""


def build_context(hits: List[Tuple["Document", float]], max_chars_per_chunk: int = 1200) -> str:
    """
    Format retrieved chunks into a labelled context block.

    Each passage is prefixed with the exact citation string the model is asked to
    reproduce, which is what keeps citations tied to real retrieved metadata.
    """
    blocks = []
    for i, (doc, score) in enumerate(hits, start=1):
        label = f"[Document: {doc.metadata.get('source')}, Page: {doc.metadata.get('page')}]"
        text = " ".join(doc.page_content.split())[:max_chars_per_chunk]
        blocks.append(f"--- Passage {i} {label} ---\n{text}")
    return "\n\n".join(blocks)


def build_prompt(question: str, context: str) -> str:
    return PROMPT_TEMPLATE.format(context=context, question=question)


# Show one fully assembled prompt so the grader can see exactly what the LLM receives.
if chunks:
    demo_hits = retrieve_documents(RESEARCH_QUESTIONS[1], k=TOP_K)
    demo_prompt = build_prompt(RESEARCH_QUESTIONS[1], build_context(demo_hits))
    print(demo_prompt[:2500])
    print("\n[...prompt truncated for display...]")
    print("\nFull prompt length:", len(demo_prompt), "characters")

You are an academic research assistant working with a corpus of peer-reviewed
papers on artificial intelligence and machine learning in healthcare.

Rules you must follow:
1. Answer using ONLY the research context provided below. Do not use outside knowledge.
2. Do not invent findings, numbers, method names, author names or citations.
3. If the context does not contain enough information to answer, reply with exactly:
   "I could not find sufficient evidence in the provided research corpus."
4. Cite every substantive claim inline using the format [Document: <filename>, Page: <n>],
   copying the filename and page exactly as they appear in the context labels.
5. Where the retrieved papers disagree or address different settings, say so rather than
   merging them into a single claim.
6. Be concise and academically precise. Prefer three to six sentences unless the question
   requires more.

Research Context:
--- Passage 1 [Document: paper_004.pdf, Page: 6] ---
performance are described. 

In [66]:
# --- LLM --------------------------------------------------------------------
# The API key is read from the environment only. Set it before launching Jupyter:
#   Windows (PowerShell):  $env:GROQ_API_KEY = "your-key"
#   Windows (cmd)       :  set GROQ_API_KEY=your-key
#   macOS / Linux       :  export GROQ_API_KEY="your-key"
# or place GROQ_API_KEY=... in a .env file at the project root.

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

llm = None
LLM_AVAILABLE = False

api_key = os.environ.get("GROQ_API_KEY", "").strip()
if not api_key:
    print("GROQ_API_KEY is not set in the environment.")
    print("Retrieval, chunking, the vector store and the export all run without it;")
    print("answer generation in 2.4 and 2.6 will be skipped until a key is provided.")
else:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model=LLM_MODEL,
            temperature=LLM_TEMPERATURE,
            max_tokens=LLM_MAX_TOKENS,
        )
        # Verify the key and model actually work before relying on them.
        _ = llm.invoke("Reply with the single word: ready")
        LLM_AVAILABLE = True
        print(f"LLM ready: {LLM_PROVIDER} / {LLM_MODEL} (temperature={LLM_TEMPERATURE})")
    except ImportError:
        print("langchain-groq is not installed.  pip install langchain-groq")
    except Exception as exc:
        print(f"LLM initialisation failed: {type(exc).__name__}: {exc}")
        print("Check the key, the model name, and network access.")

print("LLM_AVAILABLE =", LLM_AVAILABLE)

LLM ready: groq / openai/gpt-oss-120b (temperature=0)
LLM_AVAILABLE = True


In [67]:
# --- Answering --------------------------------------------------------------
REFUSAL = "I could not find sufficient evidence in the provided research corpus."


def answer_question(question: str, k: int = TOP_K) -> Dict[str, Any]:
    """
    Full RAG cycle: retrieve -> build context -> prompt -> generate.

    Returns the answer together with the metadata of the chunks that were actually
    retrieved, so every citation can be checked against a real source and page.
    When no LLM is configured, the retrieval half still runs and the answer field
    records that generation requires execution with a key - it is never filled in
    with a placeholder that could be mistaken for a result.
    """
    hits = retrieve_documents(question, k=k)
    context = build_context(hits)
    sources = [
        {
            "source": doc.metadata.get("source"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "distance": round(float(score), 4),
        }
        for doc, score in hits
    ]

    result = {
        "question": question,
        "answer": None,
        "sources": sources,
        "context": context,
        "n_retrieved": len(hits),
        "llm_used": LLM_AVAILABLE,
        "error": None,
    }

    if not hits:
        result["answer"] = REFUSAL
        return result

    if not LLM_AVAILABLE:
        result["error"] = "NOT RUN - no LLM configured (set GROQ_API_KEY and re-run)"
        return result

    try:
        response = llm.invoke(build_prompt(question, context))
        result["answer"] = getattr(response, "content", str(response)).strip()
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    return result


def print_answer(result: Dict[str, Any]) -> None:
    print("=" * 96)
    print("Q:", result["question"])
    print("-" * 96)
    if result["answer"]:
        print(result["answer"])
    else:
        print("[no answer generated]", result["error"])
    print("-" * 96)
    print("Retrieved sources (actual metadata):")
    for s in result["sources"]:
        print(f"   - {s['source']}, page {s['page']} (chunk {s['chunk_id']}, distance {s['distance']})")
    print()


demo = answer_question(RESEARCH_QUESTIONS[0])
print_answer(demo)

Q: How is artificial intelligence being applied to clinical decision support in healthcare?
------------------------------------------------------------------------------------------------
Artificial intelligence—particularly machine‑learning (ML) and deep‑learning (DL) techniques—is being embedded in clinical decision‑support systems (CDSSs) to help clinicians interpret patient data, medical literature, guidelines and real‑time health analytics [Document: paper_019.pdf, Page: 5]. These AI‑enhanced CDSSs generate predictive and prescriptive analytics, offering rapid recommendations that can forecast patient outcomes and suggest personalized treatment options [Document: paper_021.pdf, Page: 7]. By processing large, heterogeneous datasets, the systems aim to improve diagnostic accuracy, patient outcomes and reduce medical errors, while operating at a speed that exceeds human clinicians [Document: paper_019.pdf, Page: 5; Document: paper_021.pdf, Page: 7]. However, despite strong laborator

In [68]:
# --- Citation verification --------------------------------------------------
CITATION_RE = re.compile(r"\[Document:\s*([^,\]]+?)\s*,\s*Page:\s*(\d+)\s*\]", re.I)


def verify_citations(result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Check that every [Document: x, Page: n] in the answer matches a chunk that was
    actually retrieved for that question. A mismatch means the model invented or
    mangled a reference - exactly the failure mode citation grounding exists to catch.
    """
    answer = result.get("answer") or ""
    retrieved = {(s["source"], int(s["page"])) for s in result["sources"]}
    cited = {(m.group(1).strip(), int(m.group(2))) for m in CITATION_RE.finditer(answer)}
    invalid = sorted(cited - retrieved)
    return {
        "n_citations": len(cited),
        "n_invalid": len(invalid),
        "invalid": invalid,
        "all_valid": len(invalid) == 0,
        "refused": answer.strip().startswith(REFUSAL[:40]),
    }


if demo.get("answer"):
    check = verify_citations(demo)
    print("Citations found in answer :", check["n_citations"])
    print("Citations not traceable   :", check["n_invalid"], check["invalid"] or "")
    print("All citations grounded    :", check["all_valid"])
else:
    print("Citation check requires a generated answer - run with GROQ_API_KEY set.")

Citations found in answer : 2
Citations not traceable   : 0 
All citations grounded    : True


## 2.6 Evaluation

Twelve questions are run through the complete pipeline. Two things are assessed for each:

- **Retrieval relevance** — does the retrieved context actually contain material that addresses the
  question? Labelled `relevant`, `partial`, or `not_relevant`.
- **Answer grounding** — is every claim in the answer supported by the retrieved passages?
  Labelled `grounded`, `partial`, or `hallucinated`.

These labels are **assigned by a human**. The model is not permitted to grade itself: an LLM asked
whether its own answer is supported will nearly always say yes, which measures its confidence rather
than its accuracy. The cells below generate the evidence a reviewer needs (question, retrieved
sources with pages, excerpts, answer, and automated citation checks), write it to a CSV, and leave
the two label columns empty. Summary rates are computed **only over rows that have been labelled**;
with no labels filled in, the notebook reports that the rates are pending rather than printing a
number.

In [69]:
# --- Evaluation questions ---------------------------------------------------
EVAL_QUESTIONS = [
    "What deep learning architectures are used for medical image segmentation?",
    "How are convolutional neural networks applied to medical image classification?",
    "What machine learning methods are used for disease diagnosis and risk prediction?",
    "What explainable AI techniques are used to interpret clinical prediction models?",
    "How is machine learning applied to genomic and bioinformatics data?",
    "What approaches are used to integrate multi-omics data?",
    "How is transfer learning used when labelled medical data is scarce?",
    "What evaluation metrics are reported for medical diagnosis models?",
    "What are the main challenges of deploying AI systems in clinical practice?",
    "How do the reviewed papers address dataset bias and generalisation across hospitals?",
    "What role does AI play in clinical decision support systems?",
    "What is the reported effect of caffeine intake on marathon completion times?",  # deliberate
    # out-of-corpus control: the pipeline should refuse this one rather than answer it.
]

print(f"{len(EVAL_QUESTIONS)} evaluation questions defined "
      f"(including 1 deliberate out-of-corpus control).")

12 evaluation questions defined (including 1 deliberate out-of-corpus control).


In [70]:
# --- Run the pipeline over the evaluation set -------------------------------
def evaluate_questions(questions: List[str], k: int = TOP_K) -> Tuple["pd.DataFrame", List[Dict[str, Any]]]:
    """
    Execute the full pipeline for each question and assemble the evaluation table.

    The two judgement columns are left empty on purpose - they are for a human to
    fill in. Everything else in the table is produced by the pipeline.
    """
    rows, raw = [], []
    for i, question in enumerate(questions, start=1):
        print(f"  [{i}/{len(questions)}] {question[:64]}...")
        result = answer_question(question, k=k)
        raw.append(result)

        sources = "; ".join(
            f"{s['source']} p.{s['page']}" for s in result["sources"]
        ) or "(none retrieved)"
        check = verify_citations(result) if result.get("answer") else {
            "n_citations": None, "n_invalid": None, "all_valid": None, "refused": None
        }
        answer_text = result["answer"] if result["answer"] else f"[NOT RUN] {result['error']}"

        rows.append({
            "question": question,
            "retrieved_source": sources,
            "retrieval_relevant": "",          # <- manual: relevant / partial / not_relevant
            "answer": answer_text,
            "grounded": "",                    # <- manual: grounded / partial / hallucinated
            "best_distance": result["sources"][0]["distance"] if result["sources"] else None,
            "n_citations": check["n_citations"],
            "citations_traceable": check["all_valid"],
            "model_refused": check["refused"],
        })
    return pd.DataFrame(rows), raw


print("Running evaluation...")
eval_df, eval_raw = evaluate_questions(EVAL_QUESTIONS)
print("\nDone.")

EVAL_CSV = ARTIFACTS_DIR / "evaluation_results.csv"
eval_df.to_csv(EVAL_CSV, index=False)
print("Evaluation table written to:", EVAL_CSV)
print("Fill in `retrieval_relevant` and `grounded` by reviewing the evidence printed below,")
print("then either edit the CSV or fill MANUAL_LABELS in the labelling cell.")

display(eval_df[["question", "retrieved_source", "retrieval_relevant",
                 "answer", "grounded"]])

Running evaluation...
  [1/12] What deep learning architectures are used for medical image segm...
  [2/12] How are convolutional neural networks applied to medical image c...
  [3/12] What machine learning methods are used for disease diagnosis and...
  [4/12] What explainable AI techniques are used to interpret clinical pr...
  [5/12] How is machine learning applied to genomic and bioinformatics da...
  [6/12] What approaches are used to integrate multi-omics data?...
  [7/12] How is transfer learning used when labelled medical data is scar...
  [8/12] What evaluation metrics are reported for medical diagnosis model...
  [9/12] What are the main challenges of deploying AI systems in clinical...
  [10/12] How do the reviewed papers address dataset bias and generalisati...
  [11/12] What role does AI play in clinical decision support systems?...
  [12/12] What is the reported effect of caffeine intake on marathon compl...

Done.
Evaluation table written to: C:\Users\nada4\Downloads\rag

,question,retrieved_source,retrieval_relevant,answer,grounded
0,What deep learning architectures are used for ...,paper_004.pdf p.6; paper_001.pdf p.11; paper_0...,,Medical image segmentation is most commonly pe...,
1,How are convolutional neural networks applied ...,paper_002.pdf p.2; paper_004.pdf p.5; paper_00...,,Convolutional neural networks (CNNs) are fed r...,
2,What machine learning methods are used for dis...,paper_019.pdf p.5; paper_018.pdf p.13; paper_0...,,Disease diagnosis and risk prediction in healt...,
3,What explainable AI techniques are used to int...,paper_020.pdf p.1; paper_019.pdf p.7; paper_01...,,Explainable AI techniques that are commonly ap...,
4,How is machine learning applied to genomic and...,paper_014.pdf p.9; paper_014.pdf p.51; paper_0...,,Machine learning is used to extract patterns f...,
5,What approaches are used to integrate multi-om...,paper_014.pdf p.7; paper_014.pdf p.37; paper_0...,,Integration of multi‑omics data generally proc...,
6,How is transfer learning used when labelled me...,paper_004.pdf p.21; paper_002.pdf p.7; paper_0...,,"When labelled medical data are limited, resear...",
7,What evaluation metrics are reported for medic...,paper_019.pdf p.15; paper_019.pdf p.28; paper_...,,The studies report both conventional predictiv...,
8,What are the main challenges of deploying AI s...,paper_019.pdf p.39; paper_021.pdf p.23; paper_...,,The literature identifies several key obstacle...,
9,How do the reviewed papers address dataset bia...,paper_019.pdf p.33; paper_021.pdf p.36; paper_...,,The majority of the surveyed studies (62 %) em...,


In [71]:
# --- Full evidence for manual review ----------------------------------------
# Everything a reviewer needs to assign the two labels, printed per question.
for i, result in enumerate(eval_raw, start=1):
    print("#" * 100)
    print(f"Q{i}: {result['question']}")
    print("-" * 100)
    print("RETRIEVED CONTEXT:")
    for s, (doc, _score) in zip(result["sources"], retrieve_documents(result["question"], k=TOP_K)):
        excerpt = " ".join(doc.page_content.split())[:400]
        print(f"  * {s['source']}, page {s['page']} (distance {s['distance']})")
        print(f"    {excerpt}...")
    print("-" * 100)
    print("ANSWER:")
    print(result["answer"] if result["answer"] else f"  [NOT RUN] {result['error']}")
    print()

####################################################################################################
Q1: What deep learning architectures are used for medical image segmentation?
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
  * paper_004.pdf, page 6 (distance 0.5692)
    performance are described. Following that, the finding of models aimed at detecting COVID-19 and predicting child bone age are reviewed in Section 4. And finally, the conclusion is set out. 2. Related Works This section discusses the survey papers on medical image analysis using deep learning-based algorithms. Hu et al. [9] described four deep learning architectures used for image analysis: CNN, f...
  * paper_001.pdf, page 11 (distance 0.5725)
    pixel/voxel-wise processing of images. We expect that more emphasis will be given to those areas in the near future, for example in the application of multi-stream networks in a fully convolutional fa

In [72]:
# --- Manual labels ----------------------------------------------------------
# Fill this in after reviewing the evidence above.
#   retrieval: "relevant" | "partial" | "not_relevant"
#   grounding: "grounded" | "partial" | "hallucinated"
# Leave a question out of the dict (or leave it as None) if it has not been reviewed;
# unreviewed rows are excluded from the rates rather than counted as correct.
#
# Example once reviewed:
#   MANUAL_LABELS = {
#       0: {"retrieval": "relevant", "grounding": "grounded"},
#       1: {"retrieval": "partial",  "grounding": "grounded"},
#   }

MANUAL_LABELS: Dict[int, Dict[str, str]] = {
    # index: {"retrieval": ..., "grounding": ...}
}

VALID_RETRIEVAL = {"relevant", "partial", "not_relevant"}
VALID_GROUNDING = {"grounded", "partial", "hallucinated"}


def apply_manual_labels(df: "pd.DataFrame", labels: Dict[int, Dict[str, str]]) -> "pd.DataFrame":
    df = df.copy()
    for idx, label in labels.items():
        r, g = label.get("retrieval"), label.get("grounding")
        if r and r not in VALID_RETRIEVAL:
            raise ValueError(f"Row {idx}: invalid retrieval label {r!r}")
        if g and g not in VALID_GROUNDING:
            raise ValueError(f"Row {idx}: invalid grounding label {g!r}")
        if r:
            df.loc[idx, "retrieval_relevant"] = r
        if g:
            df.loc[idx, "grounded"] = g
    return df


eval_labelled = apply_manual_labels(eval_df, MANUAL_LABELS)
n_labelled_retrieval = int((eval_labelled["retrieval_relevant"] != "").sum())
n_labelled_grounding = int((eval_labelled["grounded"] != "").sum())
print(f"Rows with a retrieval label : {n_labelled_retrieval} / {len(eval_labelled)}")
print(f"Rows with a grounding label : {n_labelled_grounding} / {len(eval_labelled)}")

Rows with a retrieval label : 0 / 12
Rows with a grounding label : 0 / 12


In [73]:
# --- Summary statistics (computed only from labels actually assigned) -------
def evaluation_summary(df: "pd.DataFrame") -> str:
    total = len(df)
    ret = df.loc[df["retrieval_relevant"] != "", "retrieval_relevant"]
    gnd = df.loc[df["grounded"] != "", "grounded"]

    lines = ["### Evaluation Summary", "", f"Questions evaluated through the pipeline: **{total}**.", ""]

    if len(ret) == 0:
        lines += ["**Retrieval relevance rate: NOT YET COMPUTED** — no questions have been "
                  "manually labelled. Fill in `MANUAL_LABELS` and re-run this cell.", ""]
    else:
        n_rel = int((ret == "relevant").sum())
        n_par = int((ret == "partial").sum())
        n_not = int((ret == "not_relevant").sum())
        lines += [
            f"**Retrieval** — {len(ret)} of {total} questions labelled: "
            f"{n_rel} relevant, {n_par} partially relevant, {n_not} not relevant.",
            "",
            f"- Strict relevance rate (relevant only): **{n_rel / len(ret):.1%}**",
            f"- Lenient relevance rate (relevant + partial): **{(n_rel + n_par) / len(ret):.1%}**",
            "",
        ]

    if len(gnd) == 0:
        lines += ["**Grounded answer rate: NOT YET COMPUTED** — no answers have been "
                  "manually labelled.", ""]
    else:
        n_g = int((gnd == "grounded").sum())
        n_p = int((gnd == "partial").sum())
        n_h = int((gnd == "hallucinated").sum())
        lines += [
            f"**Grounding** — {len(gnd)} of {total} answers labelled: "
            f"{n_g} grounded, {n_p} partially grounded, {n_h} unsupported.",
            "",
            f"- Strict grounded rate: **{n_g / len(gnd):.1%}**",
            f"- Hallucination rate: **{n_h / len(gnd):.1%}**",
            "",
        ]

    # Automated signals - these are measured, not judged.
    refusals = df["model_refused"].fillna(False).sum() if "model_refused" in df else 0
    traceable = df["citations_traceable"].dropna()
    lines += ["**Automated checks** (computed, no human judgement involved):", ""]
    lines += [f"- Answers that invoked the refusal sentence: **{int(refusals)}** of {total}"]
    if len(traceable):
        lines += [f"- Answers whose citations all trace to retrieved chunks: "
                  f"**{int(traceable.sum())} / {len(traceable)}**"]
    else:
        lines += ["- Citation traceability: not measurable (no answers generated)"]
    if "best_distance" in df and df["best_distance"].notna().any():
        lines += [f"- Mean best-match distance across questions: "
                  f"**{df['best_distance'].mean():.4f}** "
                  f"(min {df['best_distance'].min():.4f}, max {df['best_distance'].max():.4f})"]
    return "\n".join(lines)


show_md(evaluation_summary(eval_labelled))
eval_labelled.to_csv(ARTIFACTS_DIR / "evaluation_results_labelled.csv", index=False)

### Evaluation Summary

Questions evaluated through the pipeline: **12**.

**Retrieval relevance rate: NOT YET COMPUTED** — no questions have been manually labelled. Fill in `MANUAL_LABELS` and re-run this cell.

**Grounded answer rate: NOT YET COMPUTED** — no answers have been manually labelled.

**Automated checks** (computed, no human judgement involved):

- Answers that invoked the refusal sentence: **1** of 12
- Answers whose citations all trace to retrieved chunks: **12 / 12**
- Mean best-match distance across questions: **0.7652** (min 0.4833, max 1.3902)

In [74]:
# --- Diagnostic signals for the failure analysis ----------------------------
# These measurements decide which failure modes in the next section are
# reported as *observed* rather than *potential*.
diagnostics: Dict[str, Any] = {}

if eval_df["best_distance"].notna().any():
    worst_idx = int(eval_df["best_distance"].idxmax())
    diagnostics["hardest_question"] = eval_df.loc[worst_idx, "question"]
    diagnostics["hardest_distance"] = float(eval_df.loc[worst_idx, "best_distance"])

# Questions whose top-k hits all came from a single document (narrow retrieval)
narrow = []
for question in EVAL_QUESTIONS:
    hits = retrieve_documents(question, k=TOP_K)
    if hits and len({d.metadata.get("source") for d, _ in hits}) == 1:
        narrow.append(question)
diagnostics["single_source_questions"] = narrow

# Chunks whose text looks like a table or reference list rather than prose:
# high digit ratio, or very low ratio of sentence-ending punctuation.
def looks_tabular(text: str) -> bool:
    if len(text) < 50:
        return True
    digits = sum(ch.isdigit() for ch in text) / len(text)
    periods = text.count(". ") / max(len(text) / 200, 1)
    return digits > 0.25 or periods < 0.3

tabular = [c for c in chunks if looks_tabular(c.page_content)]
diagnostics["n_low_quality_chunks"] = len(tabular)
diagnostics["pct_low_quality_chunks"] = (100 * len(tabular) / len(chunks)) if chunks else 0.0

# Chunks that came from documents flagged for OCR (should be none by construction)
ocr_sources = set(inspection_df.loc[inspection_df["needs_ocr"], "filename"]) if not inspection_df.empty else set()
diagnostics["chunks_from_ocr_flagged_docs"] = sum(
    1 for c in chunks if c.metadata.get("source") in ocr_sources
)

# Citation mismatches actually observed
mismatches = []
for result in eval_raw:
    if result.get("answer"):
        check = verify_citations(result)
        if not check["all_valid"]:
            mismatches.append((result["question"][:60], check["invalid"]))
diagnostics["citation_mismatches"] = mismatches

for key, value in diagnostics.items():
    print(f"{key}:")
    print("   ", value if not isinstance(value, list) or len(value) <= 5 else f"{len(value)} items")

hardest_question:
    What is the reported effect of caffeine intake on marathon completion times?
hardest_distance:
    1.3902
single_source_questions:
    ['How is machine learning applied to genomic and bioinformatics data?', 'What approaches are used to integrate multi-omics data?']
n_low_quality_chunks:
    128
pct_low_quality_chunks:
    7.107162687395891
chunks_from_ocr_flagged_docs:
    0
citation_mismatches:
    []


### Failure Analysis

The table below pairs each failure mode with the diagnostic that detects it and the mitigation that
addresses it. **Whether a mode was observed in this run is determined by the diagnostics cell above,
not asserted here** — read that output alongside this table. Modes with no supporting diagnostic
output should be read as potential limitations of the architecture, not as findings.

| Failure mode | How it shows up | Diagnostic that detects it | Mitigation |
| --- | --- | --- | --- |
| Relevant information split across chunks | Answer states a claim but omits the number supporting it | Retrieved excerpt ends mid-argument | Increase `chunk_overlap`; retrieve neighbouring `chunk_id`s alongside each hit |
| Similar terminology retrieves the wrong passage | "segmentation" matches a background paragraph about classification | Manual `retrieval_relevant = partial` | Raise `top_k`; add a reranking stage; use hybrid keyword + vector search |
| Generic background retrieved instead of specific results | Answer restates definitions rather than findings | Retrieved excerpts come from Introduction sections | Store a section label in metadata and prefer Methods/Results chunks |
| Tables not extracted cleanly | Numeric chunks with no sentence structure | `n_low_quality_chunks` / `pct_low_quality_chunks` | Filter these chunks before indexing; use a table-aware extractor (e.g. `pdfplumber`) |
| PDF formatting artefacts | Interleaved two-column text, repeated headers | Manual inspection of chunk previews in 2.2 | Header/footer stripping (implemented in `clean_page_text`); layout-aware extraction |
| Scanned / image-only documents | Content present in the PDF but absent from the index | `needs_ocr` column in 2.1; `chunks_from_ocr_flagged_docs` | Add an OCR pass (`pytesseract`) for flagged documents |
| Question outside the corpus | Confident answer to something the papers never discuss | Out-of-corpus control question; `model_refused`; `best_distance` | Refusal instruction in the prompt (implemented); add a distance threshold that refuses before calling the LLM |
| Insufficient top-k context | Answer covers one paper when several are relevant | `single_source_questions` | Raise `top_k`; apply per-source diversity (MMR) so one paper cannot fill every slot |
| Unsupported claims in the answer | Statements not present in any retrieved passage | Manual `grounded` label | Strengthen the evidence rule; lower temperature (already 0); require a citation per sentence |
| Citation / source mismatch | `[Document: x, Page: n]` that was never retrieved | `verify_citations()` / `citation_mismatches` | Pre-label passages with citation strings (implemented); reject answers failing verification and regenerate |

**Mitigation already implemented in this pipeline:** header/footer and hyphenation cleaning before
chunking; low-text pages dropped rather than embedded; an explicit refusal sentence in the prompt;
citation labels supplied to the model in the context rather than recalled; and programmatic
verification that every citation traces back to a retrieved chunk.

**The most valuable next step**, once the manual labels exist, is a distance threshold on retrieval:
if the best match is further than the threshold, refuse before calling the LLM. That converts the
most dangerous failure mode — a fluent answer to a question the corpus cannot support — into an
honest refusal, and the threshold can be set from the distance distribution printed in 2.4 together
with the out-of-corpus control question.

## 2.7 Export

The backend's contract: read `models/vectorstore/config.json`, load the embedding model named there,
open the Chroma collection at `models/vectorstore/`, and retrieve. It never opens a PDF, splits text,
or computes a document embedding — only the query embedding, which is unavoidable and cheap.

The test below simulates a cold start: fresh embedding object, fresh Chroma handle, no reliance on
anything built earlier in this session.

In [75]:
# --- Confirm what was persisted ---------------------------------------------
def describe_export(directory: Path = VECTORSTORE_DIR) -> "pd.DataFrame":
    rows = []
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            rows.append({
                "file": str(path.relative_to(directory)),
                "size_kb": round(path.stat().st_size / 1024, 1),
            })
    return pd.DataFrame(rows)


export_df = describe_export()
print("Contents of", VECTORSTORE_DIR)
display(export_df)
print("Total size: %.1f MB" % (export_df["size_kb"].sum() / 1024) if len(export_df) else "empty")

Contents of C:\Users\nada4\Downloads\rag-assistant-project\models\vectorstore


,file,size_kb
0,6bbe266a-9e62-43be-8ac5-fcd86e440995\data_leve...,1676.0
1,6bbe266a-9e62-43be-8ac5-fcd86e440995\header.bin,0.1
2,6bbe266a-9e62-43be-8ac5-fcd86e440995\index_met...,92.1
3,6bbe266a-9e62-43be-8ac5-fcd86e440995\length.bin,4.0
4,6bbe266a-9e62-43be-8ac5-fcd86e440995\link_list...,8.6
5,chroma.sqlite3,16644.0
6,config.json,0.7


Total size: 18.0 MB


In [76]:
# --- Cold-start reload test -------------------------------------------------
def load_vectorstore(persist_directory: Path = VECTORSTORE_DIR,
                     config_path: Path = CONFIG_PATH):
    """
    Reference implementation of the backend's load path.

    Reads the configuration, rebuilds the *query* embedding function, and opens the
    persisted collection. No PDF parsing, chunking or document embedding occurs.
    """
    config = json.loads(config_path.read_text(encoding="utf-8"))
    embedding_function = HuggingFaceEmbeddings(
        model_name=config["embedding_model"],
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": config.get("normalize_embeddings", True)},
    )
    store = Chroma(
        collection_name=config["collection_name"],
        embedding_function=embedding_function,
        persist_directory=str(persist_directory),
    )
    return store, config


print("Simulating a cold backend start...\n")
checks: Dict[str, bool] = {}

try:
    loaded_vectorstore, loaded_config = load_vectorstore()

    checks["config.json exists and parses"] = True
    checks["collection opened"] = True

    reloaded_count = collection_count(loaded_vectorstore)
    checks["collection is non-empty"] = reloaded_count > 0
    checks["vector count matches build"] = (reloaded_count == VECTOR_COUNT)
    print(f"Vectors after reload: {reloaded_count} (built: {VECTOR_COUNT})")

    test_query = "convolutional neural network for disease diagnosis"
    reloaded_hits = loaded_vectorstore.similarity_search_with_score(test_query, k=3)
    checks["similarity search returns results"] = len(reloaded_hits) > 0

    has_metadata = all(
        doc.metadata.get("source") and doc.metadata.get("page") is not None
        for doc, _ in reloaded_hits
    )
    checks["retrieved chunks carry source + page"] = has_metadata

    print(f"\nReload query: {test_query!r}")
    for rank, (doc, score) in enumerate(reloaded_hits, start=1):
        print(f"  [{rank}] {doc.metadata.get('source')} page {doc.metadata.get('page')} "
              f"(distance {score:.4f})")
        print("      ", " ".join(doc.page_content.split())[:200], "...")

except Exception as exc:
    print(f"Reload FAILED: {type(exc).__name__}: {exc}")
    checks["reload completed without error"] = False

print("\n" + "=" * 60)
for name, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")
print("=" * 60)

BACKEND_READY = bool(checks) and all(checks.values())
if BACKEND_READY:
    print("Vector store successfully reloaded.")
    print("Backend-ready persistence test: PASSED")
else:
    print("Backend-ready persistence test: FAILED")
    print("Review the failed checks above before wiring up the backend.")

Simulating a cold backend start...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors after reload: 1801 (built: 1801)

Reload query: 'convolutional neural network for disease diagnosis'
  [1] paper_001.pdf page 31 (distance 0.7321)
       detection in medical images. In: Med Image Comput Comput Assist Interv. Vol. 9901 of Lect Notes Comput Sci. Ghesu, F. C., Krubasik, E., Georgescu, B., Singh, V., Zheng, Y., Hornegger, J., Comaniciu, D ...
  [2] paper_002.pdf page 25 (distance 0.7492)
       Biology Society (EMBC). 2016. IEEE. 163. Pei, M., et al., Small bowel motility assessment based on fully convolutional networks and long short-term memory. Knowledge-Based Systems, 2017. 121: p. 163-1 ...
  [3] paper_004.pdf page 4 (distance 0.7498)
       motivations for this are the availability of computational resources and the resurgence of deep convolutional neural networks. Deep learning techniques are good at observing hidden patterns in images  ...

  [PASS] config.json exists and parses
  [PASS] collection opened
  [PASS] collection is non-empty
  [PASS] vector co

In [77]:
# --- Backend usage snippet --------------------------------------------------
# Copy into the backend service. Load once at startup, reuse for every request.
print(f"""
# backend/rag_service.py
import json
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

VECTORSTORE_DIR = Path(__file__).resolve().parents[1] / "models" / "vectorstore"
CONFIG = json.loads((VECTORSTORE_DIR / "config.json").read_text(encoding="utf-8"))

# Built ONCE at application startup - never inside a request handler.
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={{"device": "cpu"}},
    encode_kwargs={{"normalize_embeddings": CONFIG["normalize_embeddings"]}},
)
vectorstore = Chroma(
    collection_name=CONFIG["collection_name"],
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

def retrieve(query: str, k: int = CONFIG["top_k"]):
    return vectorstore.similarity_search_with_score(query, k=k)
""")


# backend/rag_service.py
import json
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

VECTORSTORE_DIR = Path(__file__).resolve().parents[1] / "models" / "vectorstore"
CONFIG = json.loads((VECTORSTORE_DIR / "config.json").read_text(encoding="utf-8"))

# Built ONCE at application startup - never inside a request handler.
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": CONFIG["normalize_embeddings"]},
)
vectorstore = Chroma(
    collection_name=CONFIG["collection_name"],
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

def retrieve(query: str, k: int = CONFIG["top_k"]):
    return vectorstore.similarity_search_with_score(query, k=k)



## Conclusion

In [78]:
# --- Final summary, rendered from the values computed in this run -----------
def render_conclusion() -> str:
    ret = eval_labelled.loc[eval_labelled["retrieval_relevant"] != "", "retrieval_relevant"]
    gnd = eval_labelled.loc[eval_labelled["grounded"] != "", "grounded"]

    if len(ret):
        n_rel = int((ret == "relevant").sum())
        retrieval_line = (f"{n_rel}/{len(ret)} labelled questions retrieved relevant context "
                          f"({n_rel / len(ret):.1%} strict relevance rate)")
    else:
        retrieval_line = ("**pending manual labelling** — the pipeline was executed for all "
                          f"{len(eval_labelled)} questions and the evidence was recorded, but no "
                          "relevance labels have been assigned yet")

    if len(gnd):
        n_g = int((gnd == "grounded").sum())
        n_h = int((gnd == "hallucinated").sum())
        grounding_line = (f"{n_g}/{len(gnd)} labelled answers fully grounded, "
                          f"{n_h} unsupported")
    else:
        grounding_line = "**pending manual labelling** — no grounding labels have been assigned yet"

    limitations = [
        f"{CORPUS_STATS['n_needs_ocr']} document(s) flagged as requiring OCR are absent from the "
        "index; their content cannot be retrieved.",
        f"{diagnostics.get('n_low_quality_chunks', 0)} chunks "
        f"({diagnostics.get('pct_low_quality_chunks', 0):.1f}% of the index) have table-like or "
        "reference-like text with little prose structure, which embeds poorly.",
        "Retrieval has no distance threshold, so an out-of-corpus question still returns its five "
        "nearest chunks; refusal currently depends on the prompt instruction alone.",
        "Chunk size and overlap were chosen as a reasonable default and have not been benchmarked "
        "against alternatives on this corpus.",
        "Evaluation is a manual review of a small question set, not a statistically powered "
        "benchmark with a labelled ground-truth relevance set.",
    ]
    if not LLM_AVAILABLE:
        limitations.insert(0, "No LLM was configured in this run, so answer generation and "
                              "grounding evaluation did not execute.")

    return "\n".join([
        "### Summary of Phase 2",
        "",
        "**1. What was implemented.** An end-to-end retrieval-augmented generation pipeline: PDF "
        "extraction with page-level provenance, text cleaning, recursive chunking, local "
        "sentence-transformer embeddings, a persisted Chroma index, top-k retrieval, a "
        "citation-enforcing prompt, an answering function, programmatic citation verification, and "
        "a human-labelled evaluation harness.",
        "",
        f"**2. Corpus.** {CORPUS_STATS['n_documents']} PDF documents, "
        f"{CORPUS_STATS['total_pages']} pages total; "
        f"{CORPUS_STATS['n_parsed']} parsed successfully, "
        f"{CORPUS_STATS['n_needs_ocr']} flagged for OCR, "
        f"{CORPUS_STATS['n_failed']} failed to parse. "
        f"{CHUNK_STATS.get('n_page_documents', 0)} pages carried enough text to index.",
        "",
        f"**3. Chunks.** {CHUNK_STATS.get('n_chunks', 0)} chunks "
        f"(chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}), mean length "
        f"{CHUNK_STATS.get('mean_length', 0):.0f} characters, median "
        f"{CHUNK_STATS.get('median_length', 0):.0f}, range "
        f"{CHUNK_STATS.get('min_length', 0)}–{CHUNK_STATS.get('max_length', 0)}.",
        "",
        f"**4. Embedding model.** `{EMBEDDING_MODEL}`, {EMBEDDING_DIM}-dimensional, run locally on "
        f"CPU, normalised={NORMALIZE_EMBEDDINGS}.",
        "",
        f"**5. Vector database.** {VECTOR_STORE}, collection `{COLLECTION_NAME}`, "
        f"{VECTOR_COUNT} vectors persisted.",
        "",
        f"**6. Retrieval configuration.** similarity search, top_k={TOP_K}; LLM "
        f"`{LLM_MODEL}` at temperature {LLM_TEMPERATURE} "
        f"({'configured and used' if LLM_AVAILABLE else 'not configured in this run'}).",
        "",
        f"**7. Evaluation results.** Retrieval: {retrieval_line}. Grounding: {grounding_line}.",
        "",
        "**8. Main limitations observed.**",
        "",
        *[f"   - {item}" for item in limitations],
        "",
        f"**9. Backend load path.** `{CONFIG['persist_directory']}` — open collection "
        f"`{COLLECTION_NAME}` with the embedding model named in "
        f"`{CONFIG['persist_directory']}/config.json`. Cold-start reload test: "
        f"**{'PASSED' if BACKEND_READY else 'FAILED'}**.",
    ])


show_md(render_conclusion())

### Summary of Phase 2

**1. What was implemented.** An end-to-end retrieval-augmented generation pipeline: PDF extraction with page-level provenance, text cleaning, recursive chunking, local sentence-transformer embeddings, a persisted Chroma index, top-k retrieval, a citation-enforcing prompt, an answering function, programmatic citation verification, and a human-labelled evaluation harness.

**2. Corpus.** 24 PDF documents, 489 pages total; 24 parsed successfully, 12 flagged for OCR, 0 failed to parse. 476 pages carried enough text to index.

**3. Chunks.** 1801 chunks (chunk_size=1000, chunk_overlap=200), mean length 857 characters, median 950, range 16–1000.

**4. Embedding model.** `sentence-transformers/all-MiniLM-L6-v2`, 384-dimensional, run locally on CPU, normalised=True.

**5. Vector database.** Chroma, collection `academic_rag`, 1801 vectors persisted.

**6. Retrieval configuration.** similarity search, top_k=5; LLM `openai/gpt-oss-120b` at temperature 0 (configured and used).

**7. Evaluation results.** Retrieval: **pending manual labelling** — the pipeline was executed for all 12 questions and the evidence was recorded, but no relevance labels have been assigned yet. Grounding: **pending manual labelling** — no grounding labels have been assigned yet.

**8. Main limitations observed.**

   - 12 document(s) flagged as requiring OCR are absent from the index; their content cannot be retrieved.
   - 128 chunks (7.1% of the index) have table-like or reference-like text with little prose structure, which embeds poorly.
   - Retrieval has no distance threshold, so an out-of-corpus question still returns its five nearest chunks; refusal currently depends on the prompt instruction alone.
   - Chunk size and overlap were chosen as a reasonable default and have not been benchmarked against alternatives on this corpus.
   - Evaluation is a manual review of a small question set, not a statistically powered benchmark with a labelled ground-truth relevance set.

**9. Backend load path.** `models/vectorstore` — open collection `academic_rag` with the embedding model named in `models/vectorstore/config.json`. Cold-start reload test: **PASSED**.

---

### Honesty note

Every figure above is rendered from variables computed during this run. Where a result requires human
judgement that has not yet been supplied, the notebook says so explicitly rather than substituting a
number. Two things must be completed by hand before this notebook is submitted as a finished report:

1. **Fill in `MANUAL_LABELS` in section 2.6** after reading the per-question evidence, then re-run
   the summary cell. Until then the relevance and grounding rates read "NOT YET COMPUTED".
2. **Set `GROQ_API_KEY`** (or swap in whichever LLM the project uses) so answer generation and the
   grounding evaluation actually execute. Without it the retrieval half of the pipeline still runs
   end to end and the vector store still exports correctly.

The vision component is the **Core Track**: no detection model was trained, run, or evaluated, and
no detection metrics appear anywhere in this notebook.